## setup

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
from collections import defaultdict

# processed directory and base dataset on GCS (requires gcsfs: pip install gcsfs)
PROCESSED_DIR = "gs://cs229-central/data/processed/v3/"
BASE_CSV = "gs://cs229-central/data/processed/v3/base_dataset.csv"

# output directory for bootstrapped data (local)
BOOTSTRAP_DIR = Path("data/bootstrapped/v3")
BOOTSTRAP_DIR.mkdir(parents=True, exist_ok=True)

# local directory for saving metadata file
LOCAL_METADATA_DIR = Path("data/bootstrapped/v3")
LOCAL_METADATA_DIR.mkdir(parents=True, exist_ok=True)

def _gcs_npy_uri(relative_path):
    """Build full GCS URI for a file under PROCESSED_DIR."""
    p = str(relative_path).lstrip("/")
    return PROCESSED_DIR.rstrip("/") + "/" + p

def load_npy_from_gcs(relative_path):
    """Load .npy array from GCS (requires fsspec/gcsfs)."""
    import fsspec
    uri = _gcs_npy_uri(relative_path)
    with fsspec.open(uri, "rb") as f:
        return np.load(f)

print("PROCESSED_DIR:", PROCESSED_DIR)
print("BOOTSTRAP_DIR:", BOOTSTRAP_DIR)
print("BASE_CSV:", BASE_CSV)
print("LOCAL_METADATA_DIR:", LOCAL_METADATA_DIR)

PROCESSED_DIR: gs://cs229-central/data/processed/v3/
BOOTSTRAP_DIR: data/bootstrapped/v3
BASE_CSV: gs://cs229-central/data/processed/v3/base_dataset.csv
LOCAL_METADATA_DIR: data/bootstrapped/v3


## load base dataset (generated by data_processing.ipynb)

In [5]:
base_df = pd.read_csv(BASE_CSV)

n_total = base_df.shape[0]
n_multi = int(base_df["y"].sum())
n_single = n_total - n_multi

min_frames = base_df["n_frames_used"].min()

print(f"total trajectories: {n_total}")
print(f"multi-state: {n_multi} | single-state: {n_single}")
print(f"shortest early segment: {int(min_frames)} frames")

base_df.head()

total trajectories: 307
multi-state: 85 | single-state: 222
shortest early segment: 149 frames


,receptor,rep,n_frames_total,n_frames_used,early_frac,n_windows,rmsd_mean,rmsd_std,rmsd_max,rg_mean,...,tm3_tm6_std,simID,traj_file,top_file,label,dominant_cluster_frac,silhouette,y,early_ts_path,early_frames_path
0,5-hydroxytryptamine_receptor_1B~4IAQ_A,1,2500,1200,0.5,1,4.474691,1.174720,6.749105,22.182590,...,NaN,85,5-hydroxytryptamine_receptor_1B~4IAQ_A~unknown...,5-hydroxytryptamine_receptor_1B~4IAQ_A~85~1082...,multi_state,0.574583,0.500110,1.0,early_ts_5-hydroxytryptamine_receptor_1B_4IAQ_...,early_frames_5-hydroxytryptamine_receptor_1B_4...
1,5-hydroxytryptamine_receptor_1B~4IAR_A,1,2500,1200,0.5,1,2.547754,0.408294,4.311748,21.854927,...,NaN,87,5-hydroxytryptamine_receptor_1B~4IAR_A~unknown...,5-hydroxytryptamine_receptor_1B~4IAR_A~87~1084...,single_state,0.639167,0.426066,0.0,early_ts_5-hydroxytryptamine_receptor_1B_4IAR_...,early_frames_5-hydroxytryptamine_receptor_1B_4...
2,5-hydroxytryptamine_receptor_1B~4IAR_A,1,2500,1200,0.5,1,2.601540,0.409481,4.350486,22.018714,...,NaN,90,5-hydroxytryptamine_receptor_1B~4IAR_A~unknown...,5-hydroxytryptamine_receptor_1B~4IAR_A~90~1087...,single_state,0.789167,0.546720,0.0,early_ts_5-hydroxytryptamine_receptor_1B_4IAR_...,early_frames_5-hydroxytryptamine_receptor_1B_4...
3,5-hydroxytryptamine_receptor_2B~4IB4_A,1,2500,1200,0.5,1,2.911906,0.504053,4.357796,22.523872,...,2.253397,92,5-hydroxytryptamine_receptor_2B~4IB4_A~unknown...,5-hydroxytryptamine_receptor_2B~4IB4_A~unknown...,single_state,0.663750,0.417202,0.0,early_ts_5-hydroxytryptamine_receptor_2B_4IB4_...,early_frames_5-hydroxytryptamine_receptor_2B_4...
4,5-hydroxytryptamine_receptor_2B~4IB4_A,1,2500,1200,0.5,1,3.474734,0.516913,4.636500,22.303306,...,1.804635,94,5-hydroxytryptamine_receptor_2B~4IB4_A~unknown...,5-hydroxytryptamine_receptor_2B~4IB4_A~unknown...,multi_state,0.600417,0.539242,1.0,early_ts_5-hydroxytryptamine_receptor_2B_4IB4_...,early_frames_5-hydroxytryptamine_receptor_2B_4...


## extract metadata for bootstrapping

In [6]:
L = 149                # window length in frames (must be <= the shortest early segement)
min_gap = L // 5       # minimum num frames between each samples chunk (reduce overlap & redundancy)
n_chunks_multi = 100   # chunks per multi-state trajectory
n_chunks_single = 30   # chunks per single-state trajectory
SEED = 0

rng = np.random.default_rng(SEED)
print("rng ready:", rng)

rng ready: Generator(PCG64)


In [8]:
# === build chunk metadata ===
# for each trajectory, draw K random contiguous chunks of length L

rows = [] # dictionary of sampled chunks

for _, r in base_df.iterrows():
    label = int(r["y"]) # 0 or 1
    n_chunks = n_chunks_multi if label == 1 else n_chunks_single # number of chunks to sample depends on class

    early_ts = load_npy_from_gcs(r["early_ts_path"])  # load from GCS (n_frames, 3)
    n_frames = early_ts.shape[0]

    if n_frames < L: # just in case: should have picked L >= shortest early segment
        continue

    start_idxs = []
    attempts = 0 # randomly drawing valid chunk starts that satisfy spacing rule
    max_attempts = 10 * n_chunks # safety to avoid infinite loop

    while len(start_idxs) < n_chunks and attempts < max_attempts:
        start_idx = int(rng.integers(0, n_frames - L + 1))

        if all(abs(start_idx - prev) >= min_gap for prev in start_idxs): # check spacing constraint
            start_idxs.append(start_idx)

        attempts += 1

    for start_idx in start_idxs: # create metadata record for each chunk
        rows.append({
            "traj_id": f"{r.receptor}_rep{int(r.rep)}_{int(r.simID)}",
            "receptor": r.receptor,
            "rep": int(r.rep),
            "simID": int(r.simID),
            "label": label,
            "chunk_start": start_idx,
            "chunk_length": L
        })

bootstrap_meta = pd.DataFrame(rows)

print(f"total chunks generated: {bootstrap_meta.shape[0]}")
print(f"multi-state chunks: {bootstrap_meta['label'].sum()}")
print(f"single-state chunks: {bootstrap_meta.shape[0] - bootstrap_meta['label'].sum()}")
print(f"positive fraction: {bootstrap_meta['label'].mean():.3f}")

bootstrap_meta.to_csv("bootstrap_metadata.csv", index=False)
print("saved metadata locally:", "bootstrap_metadata.csv")
bootstrap_meta.head()

total chunks generated: 7564
multi-state chunks: 2463
single-state chunks: 5101
positive fraction: 0.326
saved metadata locally: bootstrap_metadata.csv


,traj_id,receptor,rep,simID,label,chunk_start,chunk_length
0,5-hydroxytryptamine_receptor_1B~4IAQ_A_rep1_85,5-hydroxytryptamine_receptor_1B~4IAQ_A,1,85,1,346,149
1,5-hydroxytryptamine_receptor_1B~4IAQ_A_rep1_85,5-hydroxytryptamine_receptor_1B~4IAQ_A,1,85,1,276,149
2,5-hydroxytryptamine_receptor_1B~4IAQ_A_rep1_85,5-hydroxytryptamine_receptor_1B~4IAQ_A,1,85,1,1021,149
3,5-hydroxytryptamine_receptor_1B~4IAQ_A_rep1_85,5-hydroxytryptamine_receptor_1B~4IAQ_A,1,85,1,992,149
4,5-hydroxytryptamine_receptor_1B~4IAQ_A_rep1_85,5-hydroxytryptamine_receptor_1B~4IAQ_A,1,85,1,15,149


In [ ]:
# === save npy files for each chunk + a bootstrapped dataset csv ===

meta_rows = []

# lookup: (receptor, rep) -> early_ts_path
ts_lookup = {
    (row.receptor, int(row.simID), int(row.rep)): row.early_ts_path
    for row in base_df.itertuples(index=False)
}

# group chunks by trajectory so numbering resets per (receptor, rep)
for (receptor, simID, rep), grp in bootstrap_meta.groupby(["receptor", "simID", "rep"], sort=False):
    rep = int(rep)
    simID = int(simID)
    traj_id = f"{receptor}_rep{rep}_{simID}"

    early_ts = load_npy_from_gcs(ts_lookup[(receptor, simID, rep)])  # (n_frames, 3) from GCS

    for local_chunk_id, chunk_info in enumerate(grp.to_dict("records"), start=1):
        label = int(chunk_info["label"])
        start_idx = int(chunk_info["chunk_start"])
        chunk_length = int(chunk_info["chunk_length"])

        chunk = early_ts[start_idx : start_idx + chunk_length]  # (L, 3)

        file_name = f"{traj_id}_{local_chunk_id:03d}.npy"
        np.save(BOOTSTRAP_DIR / file_name, chunk)

        meta_rows.append({
            "traj_id": traj_id,
            "receptor": receptor,
            "simID": simID,
            "rep": rep,
            "y": label,
            "chunk_id": local_chunk_id,     # resets per trajectory
            "chunk_file": file_name,
            "chunk_start": start_idx,
            "chunk_length": chunk_length,
        })

bootstrap_df = pd.DataFrame(meta_rows)

print("saved chunks:", len(bootstrap_df))
print("multi chunks:", int(bootstrap_df["y"].sum()))
print("single chunks:", int((bootstrap_df["y"] == 0).sum()))

bootstrap_csv = LOCAL_METADATA_DIR / "bootstrap_dataset.csv"
bootstrap_df.to_csv(bootstrap_csv, index=False)

print("saved metadata locally:", bootstrap_csv, "| exists:", bootstrap_csv.exists())
bootstrap_df.head()

saved chunks: 4483
multi chunks: 2453
single chunks: 2030
saved: /home/jupyter/cs229-md-prediction/data/bootstrapped/v3/bootstrap_dataset.csv | exists: True


,traj_id,receptor,simID,rep,y,chunk_id,chunk_file,chunk_start,chunk_length
0,5-hydroxytryptamine_receptor_1B~4IAQ_A_rep1_85,5-hydroxytryptamine_receptor_1B~4IAQ_A,85,1,1,1,5-hydroxytryptamine_receptor_1B~4IAQ_A_rep1_85...,894,149
1,5-hydroxytryptamine_receptor_1B~4IAQ_A_rep1_85,5-hydroxytryptamine_receptor_1B~4IAQ_A,85,1,1,2,5-hydroxytryptamine_receptor_1B~4IAQ_A_rep1_85...,670,149
2,5-hydroxytryptamine_receptor_1B~4IAQ_A_rep1_85,5-hydroxytryptamine_receptor_1B~4IAQ_A,85,1,1,3,5-hydroxytryptamine_receptor_1B~4IAQ_A_rep1_85...,537,149
3,5-hydroxytryptamine_receptor_1B~4IAQ_A_rep1_85,5-hydroxytryptamine_receptor_1B~4IAQ_A,85,1,1,4,5-hydroxytryptamine_receptor_1B~4IAQ_A_rep1_85...,283,149
4,5-hydroxytryptamine_receptor_1B~4IAQ_A_rep1_85,5-hydroxytryptamine_receptor_1B~4IAQ_A,85,1,1,5,5-hydroxytryptamine_receptor_1B~4IAQ_A_rep1_85...,323,149


In [ ]:
BUCKET_DIR = "gs://cs229-central"

destination_dir = f"{BUCKET_DIR}/data/bootstrapped/v3"
processed_data_local_path = BOOTSTRAP_DIR 

dry_run = True

if dry_run:
    # 1. Preview what will be uploaded (no hidden files)
    !gsutil -m rsync -r -n -x '(^|\/)\..*' {processed_data_local_path} {destination_dir}
else:
    #When ready to actually upload removes the -n (no-op) flag
    !gsutil -m rsync -r -x '(^|\/)\..*' {processed_data_local_path} {destination_dir}

Building synchronization state...
Starting synchronization...
Copying file:///home/jupyter/cs229-md-prediction/data/bootstrapped/v3/5-hydroxytryptamine_receptor_1B~4IAQ_A_rep1_85_004.npy [Content-Type=application/octet-stream]...
Copying file:///home/jupyter/cs229-md-prediction/data/bootstrapped/v3/5-hydroxytryptamine_receptor_1B~4IAQ_A_rep1_85_002.npy [Content-Type=application/octet-stream]...
Copying file:///home/jupyter/cs229-md-prediction/data/bootstrapped/v3/5-hydroxytryptamine_receptor_1B~4IAQ_A_rep1_85_001.npy [Content-Type=application/octet-stream]...
Copying file:///home/jupyter/cs229-md-prediction/data/bootstrapped/v3/5-hydroxytryptamine_receptor_1B~4IAQ_A_rep1_85_005.npy [Content-Type=application/octet-stream]...
Copying file:///home/jupyter/cs229-md-prediction/data/bootstrapped/v3/5-hydroxytryptamine_receptor_1B~4IAQ_A_rep1_85_008.npy [Content-Type=application/octet-stream]...
Copying file:///home/jupyter/cs229-md-prediction/data/bootstrapped/v3/5-hydroxytryptamine_receptor